# Validate PLUMED conda packages on a Colab GPU

1. Select **Runtime → Change runtime type → GPU** and connect.
2. Choose **Runtime → Run all**. No laptop files, GitHub token or manual package installation are needed.
3. Return the automatically downloaded **`plumed-gpu-results.zip`**.

The notebook downloads the exact validated dev5 packages, verifies their SHA256 checksums,
and tests **CUDA 12.9**, **CUDA 13.0** and **OpenACC** in separate environments against the same kernel.
Each variant runs four existing numerical comparisons: automatic and explicit module loading,
with values and derivatives checked against references. OpenACC also checks GPU kernel-launch diagnostics.

**PASS requires all three variants.** An incompatible Colab driver is reported as an installation/test failure;
CUDA compatibility is never forced and host execution cannot satisfy the OpenACC GPU test.

Packages are downloaded anonymously through [nightly.link](https://nightly.link/) from pinned public
GitHub Actions artifacts, not from a moving “latest build”. These artifacts expire on **2027-02-10**.
The test source is pinned as well. Logs, hardware details and installed package versions are exported;
Conda environments and binaries are excluded from the results ZIP.


In [ ]:
import hashlib
import json
import os
import platform
import shutil
import subprocess
import time
import urllib.request
import zipfile
from datetime import datetime, timezone
from pathlib import Path

work_root = Path("/content/plumed-gpu-validation")
setup_logs = work_root / "logs"
setup_logs.mkdir(parents=True, exist_ok=True)
conda_prefix = work_root / "conda"
source_dir = work_root / "source"
channel_dir = work_root / "channel"
run_dir = None
version = "2.11.0dev5"
source_revision = "f7aa9009b402cfeadba7d750553d697893dfb5d1"
kernel_archive_sha256 = (
    "c4ac028c8badfc8a956a3eab79200c75cc7dd227eed5c5b427ec88760ddb64cf"
)
artifacts = [
    {
        "name": "conda-ubuntu-latest.zip",
        "url": "https://nightly.link/GiovanniBussi/plumed2/actions/runs/43207934707/conda-ubuntu-latest.zip",
        "sha256": "022965c26a556118e3005fd2f6f20a22633ca40bad1288a17b40e56acc7a810b",
    },
    {
        "name": "openacc-conda-ubuntu-latest.zip",
        "url": "https://nightly.link/GiovanniBussi/plumed2/actions/runs/43201389162/openacc-conda-ubuntu-latest.zip",
        "sha256": "02ff2b22ae4525400ba8a4973502d2d705c3ddb9a2c1bbea6a0a6c7f3b93bab8",
    },
]
validation_env = dict(os.environ)
for key in list(validation_env):
    if key.startswith("PLUMED_") or key in (
        "CONDA_OVERRIDE_CUDA",
        "PYTHONPATH",
        "PYTHONHOME",
    ):
        validation_env.pop(key)
validation_env.update(
    CONDA_PKGS_DIRS=str(work_root / "package-cache"),
    CONDA_REGISTER_ENVS="false",
    CONDA_NUMBER_CHANNEL_NOTICES="0",
)


def download(url, destination, expected_sha256=None):
    destination.parent.mkdir(parents=True, exist_ok=True)
    if (
        expected_sha256
        and destination.exists()
        and hashlib.sha256(destination.read_bytes()).hexdigest() == expected_sha256
    ):
        return destination
    for attempt in range(3):
        try:
            request = urllib.request.Request(
                url, headers={"User-Agent": "PLUMED-Colab-validation"}
            )
            with (
                urllib.request.urlopen(request, timeout=60) as response,
                destination.with_suffix(destination.suffix + ".part").open(
                    "wb"
                ) as output,
            ):
                shutil.copyfileobj(response, output)
            destination.with_suffix(destination.suffix + ".part").replace(destination)
            break
        except OSError as error:
            if attempt == 2:
                raise RuntimeError(
                    f"Download failed: {url}: {error}. Retry the cell. Expired GitHub Actions artifacts require a refreshed build."
                ) from None
            time.sleep(2**attempt)
    if expected_sha256:
        actual = hashlib.sha256(destination.read_bytes()).hexdigest()
        if actual != expected_sha256:
            raise RuntimeError(f"Checksum mismatch for {destination.name}: {actual}")
    return destination


def run_logged(command, log_path, *, env=None, cwd=None, check=True):
    print("Running:", " ".join(map(str, command)), flush=True)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("w") as output:
        result = subprocess.run(
            list(map(str, command)),
            env=env or validation_env,
            cwd=cwd,
            stdout=output,
            stderr=subprocess.STDOUT,
            check=False,
        )
    print("\n".join(log_path.read_text(errors="replace").splitlines()[-20:]))
    if check and result.returncode:
        raise RuntimeError(
            f"Command failed; see {log_path.name}. Run the final cell to download available logs."
        )
    return result.returncode

In [ ]:
if platform.system() != "Linux" or platform.machine() != "x86_64":
    raise RuntimeError(
        "Use a hosted Linux x86-64 Colab GPU runtime for these packages."
    )
if shutil.which("nvidia-smi") is None:
    raise RuntimeError(
        "No NVIDIA GPU runtime is connected. Select a GPU runtime and rerun."
    )
run_logged(["nvidia-smi"], setup_logs / "hardware.log")

In [ ]:
# Download and validate the original CI archives directly in Colab.
linux_channel = channel_dir / "linux-64"
linux_channel.mkdir(parents=True, exist_ok=True)
package_hashes = {}
for artifact in artifacts:
    archive_path = download(
        artifact["url"], work_root / "downloads" / artifact["name"], artifact["sha256"]
    )
    with zipfile.ZipFile(archive_path) as archive:
        for member in archive.namelist():
            path = Path(member)
            if path.suffix != ".conda":
                continue
            if (
                path.is_absolute()
                or ".." in path.parts
                or path.parts[-2:-1] != ("linux-64",)
            ):
                raise RuntimeError(f"Unexpected artifact entry: {member}")
            data = archive.read(member)
            digest = hashlib.sha256(data).hexdigest()
            target = linux_channel / path.name
            if (
                target.exists()
                and hashlib.sha256(target.read_bytes()).hexdigest() != digest
            ):
                raise RuntimeError(f"Conflicting package copy: {path.name}")
            if not target.exists():
                target.write_bytes(data)
            package_hashes[path.name] = digest
    print("Verified:", artifact["name"])

kernels = list(linux_channel.glob(f"plumed-{version}-*.conda"))
if len(kernels) != 1 or package_hashes.get(kernels[0].name) != kernel_archive_sha256:
    raise RuntimeError("The artifacts do not contain the expected common kernel.")
for build in ("cuda129", "cuda130"):
    if (
        len(
            list(
                linux_channel.glob(
                    f"plumed-ext-cuda-coordination-{version}-{build}_*.conda"
                )
            )
        )
        != 1
    ):
        raise RuntimeError(f"Missing or ambiguous {build} package.")
if len(list(linux_channel.glob(f"plumed-ext-openacc-ptm-{version}-*.conda"))) != 1:
    raise RuntimeError("Missing or ambiguous OpenACC package.")
if {p.name for p in linux_channel.glob("*.conda")} != set(package_hashes):
    raise RuntimeError("The channel contains packages outside the pinned artifact set.")
(setup_logs / "artifacts.json").write_text(
    json.dumps(
        {
            "version": version,
            "source_revision": source_revision,
            "archives": artifacts,
            "packages": package_hashes,
        },
        indent=2,
    )
    + "\n"
)
print(
    f"{len(package_hashes)} package archives verified; CUDA and OpenACC share the exact kernel."
)

In [ ]:
# Install an isolated conda; Colab's notebook Python is left in place.
conda = conda_prefix / "bin/conda"
if not conda.exists():
    installer_url = "https://github.com/conda-forge/miniforge/releases/download/26.9.4-2/Miniforge3-26.9.4-2-Linux-x86_64.sh"
    checksum_file = download(
        installer_url + ".sha256", work_root / "downloads/miniforge.sha256"
    )
    checksum = checksum_file.read_text().split()[0]
    if len(checksum) != 64 or any(c not in "0123456789abcdef" for c in checksum):
        raise RuntimeError("Invalid Miniforge checksum response.")
    installer = download(installer_url, work_root / "downloads/miniforge.sh", checksum)
    run_logged(
        ["bash", installer, "-b", "-p", conda_prefix],
        setup_logs / "conda-bootstrap.log",
    )
validation_env["PATH"] = str(conda_prefix / "bin") + os.pathsep + os.environ["PATH"]
run_logged(
    [
        conda,
        "install",
        "-y",
        "-p",
        conda_prefix,
        "--override-channels",
        "-c",
        "conda-forge",
        "python=3.13",
        "conda-build",
    ],
    setup_logs / "conda-tools.log",
)
if not (source_dir / ".git").exists():
    source_dir.mkdir(parents=True, exist_ok=True)
    run_logged(["git", "init", source_dir], setup_logs / "source-init.log")
    run_logged(
        [
            "git",
            "-C",
            source_dir,
            "remote",
            "add",
            "origin",
            "https://github.com/GiovanniBussi/plumed2.git",
        ],
        setup_logs / "source-remote.log",
    )
run_logged(
    [
        "git",
        "-C",
        source_dir,
        "fetch",
        "--depth=1",
        "--filter=blob:none",
        "origin",
        source_revision,
    ],
    setup_logs / "source-download.log",
)
run_logged(
    ["git", "-C", source_dir, "checkout", "--detach", source_revision],
    setup_logs / "source-checkout.log",
)
run_logged([conda, "index", channel_dir], setup_logs / "channel-index.log")
print("Setup complete. The next cell executes the GPU numerical tests.")

In [ ]:
# A new run directory makes this cell safe to rerun without reusing test environments.
run_dir = work_root / "runs" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S.%fZ")
run_dir.mkdir(parents=True)
test_env = dict(
    validation_env,
    VERSION=version,
    GPU_CONDA_CHANNEL=channel_dir.as_uri(),
    GPU_TEST_ROOT=str(run_dir),
)
exit_code = run_logged(
    ["bash", source_dir / "conda/run-gpu-test.sh"],
    run_dir / "runner.log",
    env=test_env,
    check=False,
)
transcript = (run_dir / "runner.log").read_text(errors="replace")
summary = {
    "version": version,
    "source_revision": source_revision,
    "exit_code": exit_code,
    "variants": {},
}
for variant in ("cuda12.9", "cuda13.0", "openacc"):
    log_file = run_dir / f"{variant}.log"
    log = log_file.read_text(errors="replace") if log_file.exists() else ""
    if variant == "openacc":
        expected = [
            f"Installed OpenACC numerical regression passed: nvidia {mode} {case}"
            for mode in ("autoload", "explicit")
            for case in ("rt-GPU-DISTANCE", "rt-GPU-saxs-chi2-deriv")
        ]
    else:
        expected = [
            f"Installed CUDA numerical regression passed: {mode} {case}"
            for mode in ("autoload", "explicit")
            for case in ("rt-double-nopbc", "rt-float-nopbc")
        ]
    missing = [line for line in expected if line not in log]
    if f"{variant}: installation failed" in transcript:
        status = "INSTALLATION FAILED"
    elif f"{variant}: GPU numerical tests passed" in transcript and not missing:
        status = "PASS"
    else:
        status = "NOT VALIDATED"
    summary["variants"][variant] = {"status": status, "missing_comparisons": missing}
    records = run_dir / variant / "conda-meta"
    packages = [json.loads(p.read_text()) for p in records.glob("*.json")]
    (run_dir / f"packages-{variant}.json").write_text(
        json.dumps(
            [{k: p[k] for k in ("name", "version", "build")} for p in packages],
            indent=2,
        )
        + "\n"
    )
summary["passed"] = exit_code == 0 and all(
    v["status"] == "PASS" for v in summary["variants"].values()
)
(run_dir / "summary.json").write_text(json.dumps(summary, indent=2) + "\n")
print("\nGPU validation results:")
for variant, result in summary["variants"].items():
    print(f"  {variant:12} {result['status']}")
print(
    "OVERALL:",
    "PASS" if summary["passed"] else "INCOMPLETE / FAILED — return the logs for review",
)

### Return the results

The next cell downloads a small ZIP with logs, GPU/driver details, package versions,
checksums and the test summary. Run it **even if an earlier cell failed**.
An installation failure can mean the Colab driver does not meet a variant's requirements;
the logs preserve the actual reason. All three variants must pass for a complete GPU validation.


In [ ]:
from google.colab import files

if run_dir is not None and not (run_dir / "summary.json").exists():
    (run_dir / "summary.json").write_text(
        json.dumps(
            {"passed": False, "status": "GPU run interrupted or incomplete"}, indent=2
        )
        + "\n"
    )
result_zip = work_root / "plumed-gpu-results.zip"
with zipfile.ZipFile(result_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(setup_logs.glob("*")):
        if path.is_file() and path.suffix in (".log", ".json"):
            archive.write(path, "setup/" + path.name)
    if run_dir is not None:
        for path in sorted(run_dir.glob("*")):
            if path.is_file() and path.suffix in (".log", ".json"):
                archive.write(path, "tests/" + path.name)
print(
    f"Results: {result_zip.stat().st_size:,} bytes; environments and binaries excluded."
)
if run_dir is None:
    print("GPU tests have not run; this ZIP contains setup diagnostics only.")
files.download(str(result_zip))